This notebook contains examples of results created by the scripts in this repository.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import yfinance as yf
import datetime
import scipy

# 1) Time series analysis

A time series of the historical closing price of gold is taken from Yahoo Finance. The log daily returns are calculated and plotted as a histogram and compared to a fitted Normal distribution.

In [ ]:
from get_price_returns import download_financial_data, calculate_log_daily_returns, plot_daily_returns_hist

In [ ]:
ticker = 'GC=F'

start_date = datetime.datetime(2019, 1, 1)
end_date = datetime.datetime(2026, 1, 1)

closing_prices = download_financial_data(ticker, start_date, end_date)

In [ ]:
closing_prices.describe()

In [ ]:
log_daily_returns = calculate_log_daily_returns(closing_prices)
log_daily_returns.describe()

In [ ]:
plot_daily_returns_hist(ticker, log_daily_returns)

# 2) Monte-Carlo mean-variance portfolio optimisation

Generate a sample of random portfolios for a given asset universe and plot the efficient frontier. The global minimum variance and tangency portfolios are labelled.

In [ ]:
import markowitz_model as mm

In [ ]:
n_trading_days = 252    # trading days in a year
n_portfolios = 1000    # number of random portfolios to generate

assets = ['AAPL', 'WMT', 'TSLA', 'GE', 'AMZN']  # asset universe
start_date = datetime.date(2019, 1, 1)
end_date = datetime.date.today()
risk_free_rate = 0.05

In [ ]:
dataset = mm.download_data(assets, start_date, end_date)
mm.show_data(dataset, 'Date', 'Closing price')

In [ ]:
log_daily_returns = mm.calculate_returns(dataset)
mm.show_statistics(log_daily_returns, n_trading_days)

In [ ]:
mm.show_data(log_daily_returns, 'Date', 'Log Return')

Expected return and volatility with an **equal-weighted** portfolio

In [ ]:
weights = np.ones(len(assets)) / len(assets)
mu, sigma, S = mm.get_portfolio_statistics(weights, log_daily_returns, n_trading_days, risk_free_rate)
print("Expected return: ", np.round(mu, 2))
print("Expected volatility: ", np.round(sigma, 2))
print("Sharpe ratio: ", np.round(S, 2))

### Generate random portfolios and plot risk-return and Sharpe ratio:

Expected portfolio return
\begin{equation}
\mu = \boldsymbol{\omega}^{T}\boldsymbol{\mu}
\end{equation}

Expected portfolio volatility
\begin{equation}
\sigma = \sqrt{\boldsymbol{\sigma}^{T}\boldsymbol{\Sigma}\boldsymbol{\sigma}}
\end{equation}

Sharpe ratio
\begin{equation}
S = \frac{\mu - r}{\sigma}
\end{equation}

In [ ]:
mm.plot_portfolios(assets, log_daily_returns, n_portfolios, n_trading_days, risk_free_rate=0.01)

# 3) Vasicek model

Price a zero-coupon bond using a Monte-Carlo simulation.

In [ ]:
from vasicek_model import price_bond_monte_carlo, plot_interest_rate_paths

In [ ]:
BUSINESS_DAYS = 252
YEARS = 3

In [ ]:
%%time
bond_price, walks, time_grid = price_bond_monte_carlo(
            principal=1000, 
            r0=0.05, 
            kappa=0.08, 
            theta=0.0225, 
            sigma=0.03,
            T=YEARS,
            num_simulations=1000,
            num_time_steps=BUSINESS_DAYS*YEARS
            )
print('Bond price: $%.2f' % bond_price)

Note that the model permits negative interest rates:

In [ ]:
plot_interest_rate_paths(walks, time_grid)

# 4) Parametric Value-at-Risk
Determine the Value-at-Risk for the S&P 500 stock index over a period of 6 months. 

The VaR is calculated by fitting historical returns to a normal distribution $N(\mu, \sigma^2)$:

\begin{equation}
\text{VaR}_{\alpha} = [\mu n - \sigma \sqrt{n} \Phi^{-1}(1-\alpha)]\cdot P
\end{equation}
where $\alpha$ is the confidence level, $\mu$ and $\sigma$ are the fitted mean and standard deviation of the Normal distribution, $\Phi^{-1}$ is the inverse of the cumulative density function (CDF) of the standard Normal distribution and $P$ is the position size.


In [ ]:
from value_at_risk import download_data, get_log_daily_returns, calculate_parametric_var_n

Retrieve price data and calculate log daily returns

In [ ]:
ticker_symbol = '^GSPC'
start_date = datetime.date(2001,1,1)
end_date = datetime.date.today()

price_data = download_data(ticker_symbol, start_date, end_date)
log_daily_returns = get_log_daily_returns(ticker_symbol, price_data)

In [ ]:
log_daily_returns.describe()

Determine the 99.5% VaR for a position of USD 1 million over a six month risk horizon

In [ ]:
position = 1e6
confidence_level = 0.995
mpor = 20*6

In [ ]:
var = calculate_parametric_var_n(position, confidence_level, log_daily_returns.mean(), log_daily_returns.std(), mpor)
print(f"To a {100*confidence_level:.2f}% confidence level, we do not expect to lost more than ${var:,.2f} over the next {mpor} days.")